# Scheduler ranking — Quadramind DNN accelerator

Charts for the scheduler-chooser results: which of the 14 scheduling policies
wins, on which workload mix, under which optimisation objective.

**Everything is precomputed.** The CSVs written by
`scripts/choose_scheduler.py` / `scripts/sched_objective_matrix.py` are
embedded in the next cell, so this notebook runs in Colab with no upload, no
`pip install`, and no access to the RTL or the simulator.

| Source CSV | What it holds |
|---|---|
| `ranked_<workload>_<goal>.csv` | full 14-policy ranking for one (workload, objective) pair |
| `objective_matrix.csv` | winner for every objective x workload-mix cell |
| `objective_recommendation.csv` | the one policy recommended per objective |

Timing columns (`mean_turnaround_us`, `throughput_tasks_per_s`, `energy_uJ`)
combine **model** cycle counts with **measured** post-synthesis Fmax
(Vivado 2025.2 OOC). `BATCHDNN_PP` has no current synthesis result and is
carried as *excluded* rather than silently dropped.

To plot fresh results instead, set `DATA_DIR` in the next cell to a folder of
your own `ranked_*.csv` files (e.g. a mounted Drive path).

## 0 · Data + setup

In [ ]:
# The scheduler-chooser CSVs, embedded as a zip. Set DATA_DIR to a
# folder of your own ranked_*.csv files to plot fresh results instead.
DATA_DIR = None

import base64, io, zipfile
from pathlib import Path

_EMBEDDED_CSV_ZIP = """
...
"""

if DATA_DIR is None:
    DATA_DIR = Path("/content/sched_chooser")
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(io.BytesIO(
            base64.b64decode("".join(_EMBEDDED_CSV_ZIP.split())))) as z:
        z.extractall(DATA_DIR)
DATA_DIR = Path(DATA_DIR)
print(f"{len(list(DATA_DIR.glob('*.csv')))} CSVs in {DATA_DIR}")

In [ ]:
from __future__ import annotations

import re
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# ---------------------------------------------------------------- palette ---
SURFACE = "#fcfcfb"
INK, INK2, MUTED = "#0b0b0b", "#52514e", "#898781"
GRID, AXIS = "#e1e0d9", "#c3c2b7"
CAT = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100",
       "#e87ba4", "#008300", "#4a3aa7", "#e34948"]
SEQ = ["#cde2fb", "#b7d3f6", "#9ec5f4", "#86b6ef", "#6da7ec", "#5598e7",
       "#3987e5", "#2a78d6", "#256abf", "#1c5cab", "#184f95", "#104281",
       "#0d366b"]
FAINT = "#c9d7e8"          # de-emphasised bar: slot-1 hue, low chroma
EXCLUDED = "#e6e5e0"

mpl.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE,
    "savefig.facecolor": SURFACE, "font.family": "sans-serif",
    "font.size": 10, "text.color": INK, "axes.labelcolor": INK2,
    "xtick.color": MUTED, "ytick.color": MUTED,
    "axes.edgecolor": AXIS, "axes.linewidth": 0.8,
    "grid.color": GRID, "grid.linewidth": 0.8, "grid.linestyle": "-",
    "axes.spines.top": False, "axes.spines.right": False,
    "figure.dpi": 110,
})

# -------------------------------------------------------------- goal specs --
# (column, human label, lower_is_better, value formatter)
GOAL = {
    "makespan":      ("makespan_cycles",         "Makespan (cycles)",        True,  "{:,.0f}"),
    "turnaround":    ("mean_turnaround_cycles",  "Mean turnaround (cycles)", True,  "{:,.0f}"),
    "turnaround_us": ("mean_turnaround_us",      "Mean turnaround (us)",     True,  "{:,.1f}"),
    "wait":          ("mean_wait_cycles",        "Mean wait (cycles)",       True,  "{:,.0f}"),
    "throughput":    ("throughput_tasks_per_s",  "Throughput (tasks/s)",     False, "{:,.0f}"),
    "area":          ("luts",                    "Area (LUTs)",              True,  "{:,.0f}"),
    "power":         ("power_dynamic_w",         "Dynamic power (W)",        True,  "{:.3f}"),
    "weighted":      ("weighted",                "Weighted score",           True,  "{:.3f}"),
    "energy":        ("energy_uJ",               "Energy (uJ)",              True,  "{:,.1f}"),
    "fmax":          ("fmax_mhz_post_synth",     "Fmax (MHz)",               False, "{:,.1f}"),
}


# ------------------------------------------------------------------ loader --
def load_ranked(data_dir) -> dict[tuple[str, str], pd.DataFrame]:
    """{(workload, goal): dataframe} for every ranked_*.csv in data_dir."""
    out = {}
    for p in sorted(Path(data_dir).glob("ranked_*.csv")):
        stem = p.stem[len("ranked_"):]
        goal = next((g for g in sorted(GOAL, key=len, reverse=True)
                     if stem.endswith("_" + g)), None)
        if goal is None:
            continue
        workload = stem[: -(len(goal) + 1)].replace("_", " ")
        df = pd.read_csv(p)
        df["excluded"] = df["rank"].astype(str).str.lower().eq("excluded")
        df["rank_n"] = pd.to_numeric(df["rank"], errors="coerce")
        out[(workload, goal)] = df
    return out


def scenarios(ranked) -> pd.DataFrame:
    rows = [{"workload": w, "goal": g, "n_ranked": int(d["rank_n"].notna().sum()),
             "winner": d.sort_values("rank_n").iloc[0]["scheduler"]}
            for (w, g), d in ranked.items()]
    return pd.DataFrame(rows).sort_values(["workload", "goal"]).reset_index(drop=True)


def _ink_on(hexcolor):
    """Readable ink for text sitting on a filled mark."""
    r, g, b = (int(hexcolor[i:i + 2], 16) / 255 for i in (1, 3, 5))
    lin = [c / 12.92 if c <= 0.04045 else ((c + 0.055) / 1.055) ** 2.4
           for c in (r, g, b)]
    lum = 0.2126 * lin[0] + 0.7152 * lin[1] + 0.0722 * lin[2]
    return "#ffffff" if lum < 0.42 else INK


def _style_x(ax):
    ax.grid(axis="x", zorder=0)
    ax.set_axisbelow(True)
    ax.spines["left"].set_color(AXIS)
    ax.spines["bottom"].set_visible(False)
    ax.tick_params(length=0)

## 1 · What scenarios are in the data

Each row is one `ranked_*.csv`: a workload mix scored under one objective.
`n_ranked` is how many of the 14 policies could be ranked — the rest lack a
synthesis result and are excluded from the hardware-derived goals.

In [ ]:
ranked = load_ranked(DATA_DIR)
scenarios(ranked)

## 2 · The ranking itself

One bar per policy, best at the top, winner in blue. The excluded policy is shown greyed rather than omitted, so the reader can see that it was considered.

In [ ]:
# ------------------------------------------------- 1. one ranking, one goal --
def plot_ranking(ranked, workload, goal, ax=None, top=None, show_excluded=True):
    """Horizontal ranking bars: best at the top, winner in slot-1 blue."""
    df = ranked[(workload, goal)].copy()
    col, label, lower_better, fmt = GOAL[goal]
    ok = df[~df["excluded"]].sort_values("rank_n")
    if top:
        ok = ok.head(top)
    ex = df[df["excluded"]] if show_excluded else df.iloc[0:0]
    plot_df = pd.concat([ok, ex])

    if ax is None:
        _, ax = plt.subplots(figsize=(7.6, 0.34 * len(plot_df) + 1.7))

    y = np.arange(len(plot_df))[::-1]
    vals = pd.to_numeric(plot_df[col], errors="coerce").fillna(0.0).to_numpy()
    colors = [EXCLUDED if e else (CAT[0] if i == 0 else FAINT)
              for i, e in enumerate(plot_df["excluded"])]

    ax.barh(y, vals, height=0.62, color=colors, zorder=3)
    ax.set_yticks(y)
    ax.set_yticklabels([f"{int(r)}. {s}" if pd.notna(r) else f"—  {s}"
                        for r, s in zip(plot_df["rank_n"], plot_df["scheduler"])],
                       fontsize=9, color=INK2)
    _style_x(ax)

    span = vals.max() if vals.max() else 1.0
    for yi, v, e in zip(y, vals, plot_df["excluded"]):
        txt = "not synthesised" if e else fmt.format(v)
        ax.text(v + span * 0.012, yi, txt, va="center", fontsize=8.5,
                color=MUTED if e else INK2)
    ax.set_xlim(0, span * 1.22)

    arrow = "lower is better" if lower_better else "higher is better"
    ax.set_xlabel(f"{label}  —  {arrow}", fontsize=9)
    win = plot_df.iloc[0]["scheduler"]
    ax.set_title(f"{workload} — ranked by {goal}   (winner: {win})",
                 fontsize=11.5, color=INK, loc="left", pad=10)
    return ax

In [ ]:
plot_ranking(ranked, "Workload mix 3", "turnaround_us");

## 3 · The same workload under every objective

This is the point of the chooser: the winner is not a property of the workload alone. Same mix, six objectives, four different winners.

In [ ]:
# ------------------------------------------ 2. same mix seen by every goal --
def plot_objective_panel(ranked, workload, goals=None, top=8):
    """Small multiples: the same workload re-ranked under each objective."""
    goals = goals or [g for (w, g) in ranked if w == workload]
    goals = [g for g in ["turnaround_us", "throughput", "area", "power",
                         "turnaround", "wait", "weighted", "makespan", "energy"]
             if g in goals]
    n = len(goals)
    ncol = min(3, n)
    nrow = int(np.ceil(n / ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(5.4 * ncol, 3.4 * nrow),
                             squeeze=False)
    for ax, g in zip(axes.ravel(), goals):
        plot_ranking(ranked, workload, g, ax=ax, top=top, show_excluded=False)
        win = ranked[(workload, g)].sort_values("rank_n").iloc[0]["scheduler"]
        ax.set_title(f"{g}   (winner: {win})", fontsize=10.5, loc="left", pad=8)
    for ax in axes.ravel()[n:]:
        ax.set_visible(False)
    sub = ("the winner depends on the objective" if n > 1
           else f"only one objective was scored for this workload")
    fig.suptitle(f"{workload} — {sub}", fontsize=12.5, x=0.008, ha="left", y=1.0)
    fig.tight_layout()
    return fig

In [ ]:
plot_objective_panel(ranked, "Workload mix 3");

## 4 · Rank stability across workloads

Does a policy hold its position when the workload changes? Dark = ranked well. `—` = not ranked in that scenario.

In [ ]:
# ------------------------------------------------- 3. rank-stability matrix --
def plot_rank_matrix(ranked, goal=None, workloads=None):
    """scheduler x scenario heatmap of rank position (1 = best)."""
    keys = [k for k in ranked
            if (goal is None or k[1] == goal)
            and (workloads is None or k[0] in workloads)]
    keys.sort(key=lambda k: (k[0], k[1]))
    if len(keys) < 2:
        raise ValueError(f"{len(keys)} scenario(s) match that filter — a "
                         "one-column heatmap is just a list; use plot_ranking")

    cols = [f"{w}\n{g}" if goal is None else w.replace("Workload ", "")
            for w, g in keys]
    scheds = list(ranked[keys[0]].sort_values("rank_n")["scheduler"])
    for k in keys[1:]:
        scheds += [s for s in ranked[k]["scheduler"] if s not in scheds]

    M = np.full((len(scheds), len(keys)), np.nan)
    for j, k in enumerate(keys):
        r = ranked[k].set_index("scheduler")["rank_n"]
        for i, s in enumerate(scheds):
            if s in r.index:
                M[i, j] = r[s]

    order = np.argsort(np.nanmean(np.where(np.isnan(M), 99, M), axis=1))
    M, scheds = M[order], [scheds[i] for i in order]

    cmap = mpl.colors.LinearSegmentedColormap.from_list("rank", SEQ[::-1])
    cmap.set_bad(EXCLUDED)

    fig, ax = plt.subplots(figsize=(1.25 * len(keys) + 3.2,
                                    0.42 * len(scheds) + 2.0))
    im = ax.imshow(np.ma.masked_invalid(M), cmap=cmap, vmin=1,
                   vmax=np.nanmax(M), aspect="auto")
    ax.set_xticks(range(len(keys)), cols, fontsize=9, color=INK2)
    ax.set_yticks(range(len(scheds)), scheds, fontsize=9, color=INK2)
    ax.tick_params(length=0)
    for sp in ax.spines.values():
        sp.set_visible(False)
    # 2px surface gap between cells
    ax.set_xticks(np.arange(-.5, len(keys), 1), minor=True)
    ax.set_yticks(np.arange(-.5, len(scheds), 1), minor=True)
    ax.grid(which="minor", color=SURFACE, linewidth=2)
    ax.tick_params(which="minor", length=0)

    for i in range(M.shape[0]):
        for j in range(M.shape[1]):
            v = M[i, j]
            if np.isnan(v):
                ax.text(j, i, "—", ha="center", va="center", fontsize=8.5,
                        color=MUTED)
            else:
                frac = (v - 1) / max(np.nanmax(M) - 1, 1)
                ax.text(j, i, f"{int(v)}", ha="center", va="center",
                        fontsize=9, color="#ffffff" if frac < 0.45 else INK)
    cb = fig.colorbar(im, ax=ax, fraction=0.025, pad=0.02, shrink=0.55)
    cb.set_label("rank (1 = best)", fontsize=9, color=INK2)
    cb.outline.set_visible(False)
    cb.ax.tick_params(length=0)
    title = ("rank of each scheduler per workload — goal: " + goal
             if goal else "rank of each scheduler per scenario")
    ax.set_title(title, fontsize=11.5, loc="left", pad=12, color=INK)
    fig.tight_layout()
    return fig

In [ ]:
plot_rank_matrix(ranked, goal="turnaround_us");

## 5 · Objective x workload — the winner grid

Straight from `objective_matrix.csv`: the winner and its value in every cell.

In [ ]:
# ------------------------------------------------ 4. objective x mix winners --
def plot_winner_matrix(matrix_csv):
    """Categorical heatmap of objective_matrix.csv: which scheduler wins where."""
    m = pd.read_csv(matrix_csv)
    piv = m.pivot(index="goal", columns="mix", values="winner")
    goals_order = [g for g in ["turnaround", "turnaround_us", "wait",
                               "throughput", "area", "power", "makespan",
                               "weighted"] if g in piv.index]
    piv = piv.loc[goals_order]
    winners = sorted(pd.unique(piv.values.ravel()))
    idx = {w: i for i, w in enumerate(winners)}
    M = np.vectorize(idx.get)(piv.values)

    cmap = mpl.colors.ListedColormap(CAT[: len(winners)])
    fig, ax = plt.subplots(figsize=(1.35 * piv.shape[1] + 4.0,
                                    0.55 * piv.shape[0] + 2.2))
    ax.imshow(M, cmap=cmap, aspect="auto", vmin=0, vmax=len(winners) - 1)
    ax.set_xticks(range(piv.shape[1]), list(piv.columns), fontsize=9.5, color=INK2)
    ax.set_yticks(range(piv.shape[0]), list(piv.index), fontsize=9.5, color=INK2)
    ax.tick_params(length=0)
    for sp in ax.spines.values():
        sp.set_visible(False)
    ax.set_xticks(np.arange(-.5, piv.shape[1], 1), minor=True)
    ax.set_yticks(np.arange(-.5, piv.shape[0], 1), minor=True)
    ax.grid(which="minor", color=SURFACE, linewidth=2)
    ax.tick_params(which="minor", length=0)

    val = m.set_index(["goal", "mix"])["value"].to_dict()
    for i, g in enumerate(piv.index):
        for j, mx in enumerate(piv.columns):
            w = piv.iloc[i, j]
            ink = _ink_on(CAT[idx[w]])
            ax.text(j, i - 0.13, w, ha="center", va="center", fontsize=9,
                    color=ink, fontweight="bold")
            ax.text(j, i + 0.22, str(val.get((g, mx), "")), ha="center",
                    va="center", fontsize=7.6, color=ink)
    handles = [mpl.patches.Patch(facecolor=CAT[idx[w]], label=w) for w in winners]
    ax.legend(handles=handles, loc="upper left", bbox_to_anchor=(1.01, 1.0),
              frameon=False, fontsize=9, title="winner",
              title_fontproperties={"size": 9})
    ax.set_title("objective x workload — which scheduler wins",
                 fontsize=11.5, loc="left", pad=12, color=INK)
    fig.tight_layout()
    return fig


def _label_points(ax, xs, ys, texts, colors, fontsize=8.5):
    """Greedy non-overlapping point labels (matplotlib has none built in)."""
    cands = [(8, 4), (8, -10), (-8, 4), (-8, -10), (0, 11), (0, -14),
             (14, 0), (-14, 0)]
    fig = ax.figure
    fig.canvas.draw()
    rend = fig.canvas.get_renderer()
    # the markers themselves are obstacles, not just the other labels
    placed = [mpl.transforms.Bbox.from_bounds(px - 6, py - 6, 12, 12)
              for px, py in ax.transData.transform(list(zip(xs, ys)))]
    for xv, yv, txt, col in zip(xs, ys, texts, colors):
        best = None
        for dx, dy in cands:
            a = ax.annotate(txt, (xv, yv), textcoords="offset points",
                            xytext=(dx, dy), fontsize=fontsize, color=col,
                            ha="left" if dx >= 0 else "right",
                            annotation_clip=False)
            bb = a.get_window_extent(renderer=rend).expanded(1.06, 1.25)
            clash = sum(bb.overlaps(p) for p in placed)
            if clash == 0:
                best = (a, bb)
                break
            a.remove()
            if best is None or clash < best[2]:
                best = (None, bb, clash, dx, dy)
        if best[0] is None:                      # every slot collided
            _, bb, _, dx, dy = best
            a = ax.annotate(txt, (xv, yv), textcoords="offset points",
                            xytext=(dx, dy), fontsize=fontsize, color=col,
                            ha="left" if dx >= 0 else "right",
                            annotation_clip=False)
            bb = a.get_window_extent(renderer=rend)
        placed.append(bb)

In [ ]:
plot_winner_matrix(DATA_DIR / "objective_matrix.csv");

## 6 · Cost of the latency

Latency against area, both on measured hardware numbers. A policy on the front is one no other policy beats on both axes at once.

In [ ]:
# ---------------------------------------------------- 5. latency-area Pareto --
def plot_pareto(ranked, workload, goal="turnaround_us",
                x="luts", y="mean_turnaround_us"):
    """Cost/benefit scatter with the Pareto front drawn through it."""
    df = ranked[(workload, goal)]
    d = df[~df["excluded"]].copy()
    d[x] = pd.to_numeric(d[x], errors="coerce")
    d[y] = pd.to_numeric(d[y], errors="coerce")
    d = d.dropna(subset=[x, y]).sort_values(x)

    front, best = [], np.inf
    for _, r in d.iterrows():
        if r[y] < best:
            front.append(r)
            best = r[y]
    front = pd.DataFrame(front)

    fig, ax = plt.subplots(figsize=(8.4, 5.4))
    ax.grid(True, zorder=0)
    ax.set_axisbelow(True)
    ax.set_yscale("log")
    ax.margins(x=0.14, y=0.14)
    front_set = set(front["scheduler"])
    if len(front) > 1:
        ax.step(front[x], front[y], where="post", color=CAT[0], linewidth=2,
                zorder=3, label="Pareto front")
    ax.scatter(d[x], d[y], s=70, color=FAINT, edgecolor=SURFACE, linewidth=2,
               zorder=4, label="dominated")
    ax.scatter(front[x], front[y], s=95, color=CAT[0], edgecolor=SURFACE,
               linewidth=2, zorder=5,
               label="Pareto-optimal" if len(front) > 1
                     else f"Pareto-optimal ({front.iloc[0]['scheduler']} "
                          "dominates on both axes)")
    _label_points(ax, d[x], d[y], d["scheduler"],
                  [INK2 if s in front_set else MUTED for s in d["scheduler"]])
    ax.set_xlabel(f"{GOAL['area'][1]}  —  lower is better", fontsize=9)
    ax.set_ylabel(f"{GOAL['turnaround_us'][1]}  —  lower is better", fontsize=9)
    ax.legend(frameon=False, fontsize=9, loc="upper left")
    ax.set_title(f"{workload} — latency vs area (both measured/derived)",
                 fontsize=11.5, loc="left", pad=10, color=INK)
    fig.tight_layout()
    return fig

In [ ]:
plot_pareto(ranked, "Workload mix 3");

## 7 · The recommendation

`objective_recommendation.csv` — one policy per objective, with how many of the six mixes it wins on and how the tie was broken.

In [ ]:
# -------------------------------------------------------- 6. recommendation --
def recommendation_table(rec_csv):
    r = pd.read_csv(rec_csv)
    return r[["goal", "recommended", "optimal_on", "stability",
              "co_optimal_set", "meaning"]]

In [ ]:
recommendation_table(DATA_DIR / "objective_recommendation.csv")

## 8 · Interactive: pick any scenario

Dropdowns over every (workload, objective) pair present in the data.

In [ ]:
try:
    from ipywidgets import interact, Dropdown
    _pairs = sorted(ranked)
    _w = Dropdown(options=sorted({w for w, _ in _pairs}), description="workload")
    _g = Dropdown(options=sorted({g for _, g in _pairs}), description="goal")

    def _show(workload, goal):
        if (workload, goal) not in ranked:
            print(f"no ranked_*.csv for ({workload}, {goal}); available goals "
                  f"for this workload: "
                  f"{sorted(g for w, g in _pairs if w == workload)}")
            return
        plot_ranking(ranked, workload, goal)
        plt.show()

    interact(_show, workload=_w, goal=_g)
except ImportError:
    print("ipywidgets unavailable — call plot_ranking(ranked, workload, goal)")

## 9 · Export every figure

Writes PNG + PDF (vector, for the write-up) for every scenario and downloads
them as a single zip.

In [ ]:
import itertools, shutil

OUTDIR = Path("/content/sched_figures")
shutil.rmtree(OUTDIR, ignore_errors=True)
OUTDIR.mkdir(parents=True)

def _save(fig, name):
    for ext in ("png", "pdf"):
        fig.savefig(OUTDIR / f"{name}.{ext}", bbox_inches="tight", dpi=200)
    plt.close(fig)

for (w, g) in sorted(ranked):
    _save(plot_ranking(ranked, w, g).figure,
          f"ranking_{w.replace(' ', '_')}_{g}")

for w in sorted({w for w, _ in ranked}):
    _save(plot_objective_panel(ranked, w), f"objectives_{w.replace(' ', '_')}")

for g in sorted({g for _, g in ranked}):
    try:
        _save(plot_rank_matrix(ranked, goal=g), f"rank_matrix_{g}")
    except ValueError:
        pass
_save(plot_winner_matrix(DATA_DIR / "objective_matrix.csv"), "winner_matrix")
_save(plot_pareto(ranked, "Workload mix 3"), "pareto_mix3")

shutil.make_archive("/content/sched_figures", "zip", OUTDIR)
print(f"{len(list(OUTDIR.iterdir()))} files ->  /content/sched_figures.zip")

try:
    from google.colab import files
    files.download("/content/sched_figures.zip")
except ImportError:
    pass